In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))


In [2]:
refine_plus_export_pool = Path("refine_plus_export_pool")
combined_db_export_pool = Path("combined_db_export_pool")
export_folder: Path = combined_db_export_pool / "pass_00"

In [3]:
import os
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools

lfs_to_join: list[LazyFrame] = [
    pl.scan_parquet(refine_plus_export_pool / table_name)
    for table_name in os.listdir(refine_plus_export_pool)
] + [
    dp.combined_atlas,
    dp.combined_mask.unique(
    subset=["i", "j", "face"], keep="first" # Sometimes duplicates can occur but are rare
    ).rename({
        "lod_level": "detection_lod_level",
        "lod_code": "detection_lod_code",
        "row_id" : "boulder_id"
    })
]

for lf in lfs_to_join:
    print(lf.collect_schema().names())

['i', 'j', 'face', 'area']
['i', 'j', 'face', 'TIR detailed_survey tri_num', 'TIR detailed_survey band depth 350', 'TIR detailed_survey band depth 440', 'TIR detailed_survey slope 1000', 'TIR detailed_survey ratio 1000', 'TIR detailed_survey sigma band depth 350', 'TIR detailed_survey sigma band depth 440', 'TIR detailed_survey sigma slope 1000', 'TIR detailed_survey sigma ratio 1000']
['i', 'j', 'face', 'TIR recona tri_num', 'TIR recona band depth 350', 'TIR recona band depth 440', 'TIR recona slope 1000', 'TIR recona ratio 1000', 'TIR recona sigma band depth 350', 'TIR recona sigma band depth 440', 'TIR recona sigma slope 1000', 'TIR recona sigma ratio 1000']
['i', 'j', 'face', 'TIR reconb tri_num', 'TIR reconb band depth 350', 'TIR reconb band depth 440', 'TIR reconb slope 1000', 'TIR reconb ratio 1000', 'TIR reconb sigma band depth 350', 'TIR reconb sigma band depth 440', 'TIR reconb sigma slope 1000', 'TIR reconb sigma ratio 1000']
['i', 'j', 'face', 'VNIR detailed_survey tri_num'

In [ ]:
ChunkingTools.join_in_chunks(
    export_folder = export_folder,
    lfs_to_join = lfs_to_join,
    chunks = QCubeChunk.generate(depth=4),
)

Joining:   2%|▏         | 2/96 [00:10<08:03,  5.15s/it]

In [ ]:
pl.scan_parquet(export_folder).filter(pl.col("i") == 1, pl.col("TIR detailed_survey TIR detailed_survey tri_num").is_not_nan()).collect()

i,j,face,area,TIR detailed_survey TIR detailed_survey tri_num,TIR detailed_survey band depth 350,TIR detailed_survey band depth 440,TIR detailed_survey slope 1000,TIR detailed_survey ratio 1000,TIR detailed_survey sigma band depth 350,TIR detailed_survey sigma band depth 440,TIR detailed_survey sigma slope 1000,TIR detailed_survey sigma ratio 1000,TIR detailed_survey tri_num,TIR recona TIR recona tri_num,TIR recona band depth 350,TIR recona band depth 440,TIR recona slope 1000,TIR recona ratio 1000,TIR recona sigma band depth 350,TIR recona sigma band depth 440,TIR recona sigma slope 1000,TIR recona sigma ratio 1000,TIR recona tri_num,TIR reconb TIR reconb tri_num,TIR reconb band depth 350,TIR reconb band depth 440,TIR reconb slope 1000,TIR reconb ratio 1000,TIR reconb sigma band depth 350,TIR reconb sigma band depth 440,TIR reconb sigma slope 1000,TIR reconb sigma ratio 1000,TIR reconb tri_num,VNIR detailed_survey VNIR detailed_survey tri_num,VNIR detailed_survey band depth,VNIR detailed_survey reflectance,VNIR detailed_survey slope1 poly,VNIR detailed_survey slope2 poly,VNIR detailed_survey sigma band depth,VNIR detailed_survey sigma reflectance,VNIR detailed_survey sigma slope1 poly,VNIR detailed_survey sigma slope2 poly,VNIR detailed_survey tri_num,VNIR recona VNIR recona tri_num,VNIR recona band depth,VNIR recona reflectance,VNIR recona slope1 poly,VNIR recona slope2 poly,VNIR recona sigma band depth,VNIR recona sigma reflectance,VNIR recona sigma slope1 poly,VNIR recona sigma slope2 poly,VNIR recona tri_num,VNIR reconc VNIR reconc tri_num,VNIR reconc band depth,VNIR reconc reflectance,VNIR reconc slope1 poly,VNIR reconc slope2 poly,VNIR reconc sigma band depth,VNIR reconc sigma reflectance,VNIR reconc sigma slope1 poly,VNIR reconc sigma slope2 poly,VNIR reconc tri_num,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z,detection_lod_level,detection_lod_code,boulder_id
u32,u32,str,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,u8,f32,f32,f32,f32,u8,str,u32
1,5804,"""negy""",0.001034,47813.0,1.000491,1.010565,1.002693,null,0.001784,0.001144,0.001007,null,47813,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,191115.0,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,39,0.006754,0.161978,-0.067567,-0.162062,null,null,null
1,5805,"""negy""",0.001042,47813.0,1.000491,1.010565,1.002693,null,0.001784,0.001144,0.001007,null,47813,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,191115.0,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,25,0.00438,0.16197,-0.067601,-0.16205,null,null,null
1,5806,"""negy""",0.001123,47813.0,1.000491,1.010565,1.002693,null,0.001784,0.001144,0.001007,null,47813,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,191115.0,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,14,0.002284,0.161957,-0.067637,-0.162041,null,null,null
1,5807,"""negy""",0.001092,47813.0,1.000491,1.010565,1.002693,null,0.001784,0.001144,0.001007,null,47813,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,191115.0,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,NaN,null,null,null,null,null,null,null,null,null,11,0.001882,0.161942,-0.067675,-0.162032,null,null,null
1,5808,"""negy""",0.001041,47813.0,1.000491,1.010565,1.002693,null,0.001784,0.001144,0.001007,null,47813,NaN,null,null,null,null,null,null,null,nu